<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大型语言模型》</a>一书的补充代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp?1" width="100px"></a>
</td>
</tr>
</table>

# 第 8 章：蒸馏推理模型以实现高效推理

本 notebook 使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 由 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F01_raschka.webp" width=600>

&nbsp;
## 8.1 推理任务的模型蒸馏简介

- 蒸馏是指使用较大 LLM（“教师”）的输出训练较小 LLM（“学生”）
- 蒸馏可能比从零开始使用 RLVR 训练更有效
  - 例如，[DeepSeek-R1 论文](https://arxiv.org/abs/2501.12948)报告称，对于较小的 DeepSeek-R1 模型变体，蒸馏获得的性能优于 RLVR
  - 具体来说，他们先训练最大的 DeepSeek R1 模型（671B 参数），再通过蒸馏用它训练较小的模型


- 蒸馏主要分为两类：硬蒸馏和软蒸馏
  1. 硬蒸馏：使用教师生成的文本训练学生（将教师输出视为真实目标）
  2. 软蒸馏：训练学生去匹配教师在词表上的概率分布（通常通过最小化由教师和学生 logits 得到的分布之间的 KL 散度）


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F02_raschka.webp" width=600>

- 1. 纯硬蒸馏：
  - 只使用教师生成的文本作为目标（这里将教师输出视为真实目标）
  - 损失：`L = CrossEntropy(y_teacher_tokens, y_student)`
  - 从技术上说，这是在合成数据上进行标准监督微调
  - 不需要访问教师 logits
  - DeepSeek-R1 蒸馏中的较小模型采用的正是这种方式：由大型 671B DeepSeek-R1（教师）生成推理轨迹，再用它们微调较小的 LLM（学生）


- 2. 纯软蒸馏
  - 直接匹配教师的输出概率分布
  - 损失：`L = KL(p_teacher || p_student)`
  - 训练时需要教师 logits（或对数概率）


- 3. 结合硬蒸馏和软蒸馏
  - 这是源自计算机视觉的经典知识蒸馏，详见论文 [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531)
  - 既使用教师实际输出的词元进行训练，也迫使学生匹配教师的完整概率分布
  - 损失：`L = CE(y_teacher_tokens, y_student) + lambda * KL(p_teacher || p_student)`，其中 `lambda` 是用于调节 KL 项贡献的超参数


- 实际应用中，硬蒸馏更常见
  - logits 通常无法获取；例如 OpenAI ChatGPT 或 Anthropic Claude 等专有系统不会公开完整词表 logits，因此无法进行经典软蒸馏
  - 即使可以获取 logits，软蒸馏也要求学生和教师 LLM 使用相同的分词方式，否则词表无法对应（因此软蒸馏更适合同一模型家族内部）
  - 请注意，即便只是生成并复用文本输出进行蒸馏，也可能违反某些服务商的政策
  - 即使 logits 可用，保存长序列的完整词表分布也会占用大量带宽和磁盘空间；生成并保存纯文本输出的成本要低得多


- 如果软蒸馏和硬蒸馏都可行，哪一种表现更好？
  - 设计得当的软蒸馏目标可以优于硬蒸馏（https://arxiv.org/abs/2306.08543）
  - 但目前尚无在现代 LLM 基准上进行的大规模直接对比，且只改变软、硬蒸馏这一项因素以确保公平
  - 数据生成策略的影响可能大于监督信号是软还是硬（https://arxiv.org/abs/2306.13649）


- 本章实现类似 DeepSeek-R1 的硬蒸馏，因为这是更常见且实用的方法。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F03_raschka.webp" width=600>

&nbsp;
## 8.2 生成推理蒸馏数据集

- 蒸馏的第一步是创建用于训练学生模型的数据集（这里是贯穿全书使用的 Qwen3 0.6B 基础模型）
- 我们将使用 MATH 数据集中与 MATH-500 测试集不重叠的 12,000 个数学问题（https://github.com/rasbt/math_full_minus_math500）
- 这与第 6、7 章用于 RLVR 的 12,000 个数学问题相同
- 将这些问题输入现有推理模型（这里是 DeepSeek-R1），并收集回复用于训练


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F04_raschka.webp" width=500>

- 第 6、7 章使用 RLVR 训练 Qwen3 基础模型生成正确解答
- 随后借助验证器将生成的解答与参考解答比较，以计算训练信号
- 蒸馏则通过比较教师解答与学生解答来计算训练信号


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F05_raschka.webp" width=600>

- 采用这种蒸馏方法，可以在训练学生模型之前，预先用教师模型生成蒸馏数据集
- 为全部 12,000 个数学问题生成教师答案会耗费大量时间和资源，因此我预先通过 OpenRouter 使用云端托管的 6710 亿参数 DeepSeek-R1 模型生成了数据集（API 成本约为 50 美元）
- 下一节将加载这个预生成数据集，因此无需自行生成
- 如果感兴趣，可以在[附加材料](https://github.com/rasbt/reasoning-from-scratch/tree/main/ch08/02_generate_distillation_data)中找到所用代码和使用说明


&nbsp;
## 8.3 加载用于蒸馏的 MATH 训练数据集

- 本节加载上一节所述由 DeepSeek-R1 生成的数学问题和答案数据集。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F06_raschka.webp" width=600>

- 该数据集可从 Hugging Face 模型中心获取：https://huggingface.co/datasets/rasbt/math_distill
- 以下代码将其加载到 Python 会话中：


In [ ]:
import json
import requests
from pathlib import Path


def load_distill_data(
    local_path=None,
    partition="deepseek-r1-math-train",
    save_copy=True,
):

    if local_path is None:
        local_path = f"{partition}.json"
    local_path = Path(local_path)

    url = (
        "https://huggingface.co/datasets/rasbt/math_distill"
        "/resolve/main/data/"
        f"{partition}.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        f"MATH/{partition}.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        size_kb = local_path.stat().st_size / 1e3
        print(f"{local_path}: {size_kb:.1f} KB (cached)")
        return data

    assert partition in (
        "deepseek-r1-math-train",
        "deepseek-r1-math500",
        "qwen3-235b-a22b-math-train",
        "qwen3-235b-a22b-math500",
    )

    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
    except requests.RequestException:
        print("Using backup URL.")
        r = requests.get(backup_url, timeout=30)
        r.raise_for_status()

    data = r.json()

    if save_copy:
        with local_path.open("w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

        size_kb = local_path.stat().st_size / 1e3
        print(f"{local_path}: {size_kb:.1f} KB")

    return data

In [ ]:
math_train = load_distill_data(partition="deepseek-r1-math-train")

print("Dataset size:", len(math_train))

- 查看一个训练样本以检查其结构：

In [ ]:
from pprint import pprint

pprint(math_train[4])

In [ ]:
math_train[4].keys()

- 数据集包含四个字段：
  - `gtruth_answer`：正确的参考答案（真实答案）
  - `message_thinking`：模型写出的推理轨迹
  - `message_content`：推理轨迹之后的最终答案
- 可以使用以下代码格式化答案：


In [ ]:
def format_distilled_answer(entry):
    content = str(entry["message_content"]).strip()
    if not content:
        raise ValueError("Missing non-empty 'message_content' field.")

    thinking = str(entry["message_thinking"]).strip()
    return f"<think>{thinking}</think>\n\n{content}"

In [ ]:
print(format_distilled_answer(math_train[4]))

- 如上一章所述，`<think></think>` 词元并非必需
- 它们有助于将推理轨迹与最终答案分开
- OpenAI ChatGPT 等一些服务商和界面会对用户隐藏推理轨迹；使用并训练模型使用 `<think></think>` 词元，可以更方便地解析并隐藏这些思考轨迹


- 该数据集包含真实标签，因此如果想知道 DeepSeek-R1 在该数据集上的准确率，可以使用第 3 章的验证器进行计算
- 为方便起见，可以使用附加材料（第 3 章）中的 [evaluate_json.py](https://github.com/rasbt/reasoning-from-scratch/blob/main/ch03/02_math500-verifier-scripts/evaluate_json.py) 脚本直接从 JSON 文件计算准确率


In [ ]:
from reasoning_from_scratch.ch07 import download_from_github

_ = download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_json.py"
)

In [ ]:
!uv run evaluate_json.py \
--json_path "deepseek-r1-math-train.json" \
--gtruth_answer gtruth_answer \
--generated_text message_content

&nbsp;
## 8.4 构建训练样本

- 本节准备数据集样本用于训练。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F07_raschka.webp" width=600>

- 准备工作主要包括格式化样本并进行分词
- 在 RLVR 中，每个训练样本只使用一次，因此这些工作是在使用时即时完成的
- 蒸馏属于监督微调，多次查看相同样本会带来益处
- 因此，为避免重复工作，我们只格式化和分词一次，再保存结果以供复用


&nbsp;
### 8.4.1 加载并理解分词器

- 首先从分词器开始
- 使用“推理”变体，因为它支持第 7 章讨论的 `<think>...</think>` 词元


In [ ]:
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer
)

def load_reasoning_tokenizer(local_dir="qwen3"):
    download_qwen3_small(
        kind="reasoning", tokenizer_only=True, out_dir=local_dir
    )

    tokenizer_path = Path(local_dir) / "tokenizer-reasoning.json"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )
    
    return tokenizer


tokenizer = load_reasoning_tokenizer()

- 请注意，我们设置了 `apply_chat_template=True` 和 `add_generation_prompt=True`，这会对提示词应用额外格式；可以通过以下代码演示：

In [ ]:
prompt = "Sam is hired for a 20-day period..."

prompt_ids = tokenizer.encode(prompt)
decoded_prompt = tokenizer.decode(prompt_ids)
print(decoded_prompt)

- `<|im_start|>user` 是用户所提供提示词的起始词元
- `<|im_end|>` 是用户提示词的停止词元
- `<|im_start|>assistant` 是 LLM 答案的起始词元
- 这种格式并非必需，但它是指令/聊天微调中常见且实用的约定
- 对于答案，需要通过 `chat_wrapped=False` 禁用此包装，否则提示词与答案词元合并训练时会出现过多起始词元：


In [ ]:
answer = (
    "<think>Okay, let me try to solve "
    "this problem...</think> \\boxed{4}"
)
answer_ids = tokenizer.encode(answer, chat_wrapped=False)
decoded_answer = tokenizer.decode(answer_ids)
print(decoded_answer)

- 训练时需要组合后的词元 ID，并在答案末尾包含 `<|im_end|>` 序列结束词元：

In [ ]:
token_ids = prompt_ids + answer_ids + [tokenizer.eos_token_id]
decoded_token_ids = tokenizer.decode(token_ids)
print(decoded_token_ids)

- 再次强调，我们为了使用 `<think></think>` 词元而选择推理分词器，但这并不关键；也可以使用基础分词器
- 聊天模板同样是可选的；保留它只是因为 Qwen3 自己的推理模型采用这种格式（只有使用智能体或多轮提示词时才有影响）
- 为保持一致，评估脚本中需要使用 `--which_model "reasoning"`


&nbsp;
### 8.4.2 格式化数据集并进行分词

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F08_raschka.webp" width=400>

- 下面编写一个 `build_examples` 函数，对整个数据集应用所需格式。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F09_raschka.webp" width=600>

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt

def build_examples(data, tokenizer):
    examples = []
    skipped = 0

    for entry in data:
        try:
            # 第 1 步：编码提示词
            prompt = render_prompt(entry["problem"])
            prompt_ids = tokenizer.encode(prompt)

            # 第 2 步：编码答案
            target_answer = format_distilled_answer(entry)
            answer_ids = tokenizer.encode(
                target_answer, chat_wrapped=False
            )

            # 第 3 步：合并提示词和答案
            token_ids = (
                prompt_ids + answer_ids + [tokenizer.eos_token_id]
            )

            if len(token_ids) < 2:
                skipped += 1
                continue

            examples.append({
                "token_ids": token_ids,
                "prompt_len": len(prompt_ids),
            })
        except (KeyError, TypeError, ValueError):
            skipped += 1

    return examples, skipped

In [ ]:
examples, skipped = build_examples(math_train, tokenizer)

print("Number of examples:", len(examples))
print("Number of skipped examples:", skipped)

In [ ]:
print(tokenizer.decode(examples[4]["token_ids"]))

&nbsp;
### 8.4.3 过滤并划分数据集

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F10_raschka.webp" width=400>

In [ ]:
def compute_length(examples, answer_only=False):
    lengths = []
    for ex in examples:
        total = len(ex["token_ids"])
        length = total - ex["prompt_len"] if answer_only else total
        lengths.append(length)

    avg_len = round(sum(lengths) / len(lengths))

    shortest_len = min(lengths)
    longest_len = max(lengths)
    shortest_idx = lengths.index(shortest_len)
    longest_idx = lengths.index(longest_len)

    print(f"Average: {avg_len} tokens")
    print(f"Shortest: {shortest_len} tokens (index {shortest_idx})")
    print(f"Longest: {longest_len} tokens (index {longest_idx})")

compute_length(examples)

In [ ]:
def filter_examples_by_max_len(examples, max_len=2048):
    filtered_examples = [
        s for s in examples
        if len(s["token_ids"]) <= max_len
    ]

    print("Original:", len(examples))
    print("Filtered:", len(filtered_examples))
    print("Removed:", len(examples) - len(filtered_examples))

    return filtered_examples

filtered_examples = filter_examples_by_max_len(examples, max_len=2048)

In [ ]:
compute_length(filtered_examples)

In [ ]:
import random

rng = random.Random(123)
rng.shuffle(filtered_examples)

train_examples = filtered_examples[25:]
val_examples = filtered_examples[:25]

In [ ]:
print("Number of train examples:", len(train_examples))
print("Number of validation examples:", len(val_examples))

- 练习：检查训练集和验证集的长度，确保二者均衡。

In [ ]:
compute_length(train_examples)

In [ ]:
compute_length(val_examples)

In [ ]:
import matplotlib.pyplot as plt

train_lengths = [len(ex["token_ids"]) for ex in train_examples]
val_lengths = [len(ex["token_ids"]) for ex in val_examples]

# 验证集小得多，因此对计数进行归一化
bins = range(0, max(train_lengths + val_lengths) + 64, 64)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(train_lengths, bins=bins, density=True, alpha=0.6, label="Train")
ax.hist(val_lengths, bins=bins, density=True, alpha=0.6, label="Validation")
ax.set_xlabel("Token length")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()


&nbsp;
## 8.5 加载预训练模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F11_raschka.webp" width=600>

In [ ]:
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()

model, _ = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

&nbsp;
## 8.6 计算训练损失和验证损失

- 本节实现所谓的交叉熵损失，并在蒸馏期间将其用作训练信号
- 训练过程中也会将它应用于验证集进行评估
- 有深度学习背景的读者可能熟悉分类问题中的交叉熵；这里使用的是同一个交叉熵函数，只是预测目标为序列中的下一个词元
- 可以从第 5、6 章计算过的对数概率推导交叉熵损失


- 如第 5 章首次介绍对数概率计算时所述，对数概率会为下一个生成词元计算一个分数
- 对数概率越高，表示模型为正确词元分配的概率越高（预测越好）；对数概率越低则表示概率越低（预测越差）
- 下图复用了第 5 章的示例


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F12_raschka.webp" width=600>

- 与上图类似，第 6 章的 `sequence_logprob` 函数会计算每个词元的对数概率，然后以总和形式返回（即联合对数概率）。

- 使用训练集中的一个样本来查看具体过程（为减少计算时间，选择最短的样本）：

In [ ]:
# 5730 是 train_examples 中最短的样本
token_ids = train_examples[5730]["token_ids"]
prompt_len = train_examples[5730]["prompt_len"]

- 请注意，我们没有报告 `sequence_logprob` 返回的对数概率之和，而是对答案词元数量取平均；答案词元数可用包含提示词和答案的总词元数减去 `prompt_len` 得到：

In [ ]:
from reasoning_from_scratch.ch06 import sequence_logprob

tok = torch.tensor(token_ids, dtype=torch.long, device=device)

with torch.no_grad():
    seq_logprob = sequence_logprob(model, tok, prompt_len)
    num_answer_tokens = tok.numel() - prompt_len
    avg_logprob = -seq_logprob / num_answer_tokens
print(f"Average logprob: {avg_logprob:.2f}")

- 使用 `cross_entropy` 函数可以完成相同的计算
- 该函数原本用于分类，因此除了模型 logits 外，还要求传入类别标签；这里的类别标签就是“正确”词元
- 具体来说，只对答案词元计算交叉熵（忽略提示词词元）


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F13_raschka.webp" width=600>

In [ ]:
input_ids = tok[:-1].unsqueeze(0)
target_ids = tok[1:]  # 目标是向后移动一个位置的输入
logits = model(input_ids).squeeze(0)

# 移除提示词词元
first_answer_logit_idx = max(prompt_len - 1, 0)
answer_logits = logits[first_answer_logit_idx:]
answer_targets = target_ids[first_answer_logit_idx:]

with torch.no_grad():
    ce_mean_direct = torch.nn.functional.cross_entropy(
        answer_logits, answer_targets
    )
print(f"Cross-entropy: {ce_mean_direct:.2f}")

- 可以看到，结果与之前对答案词元的对数概率取平均时相同
- 换言之，`cross_entropy` 函数内部实现了平均对数概率计算
- 使用 `cross_entropy` 而不是自行实现的 `sequence_logprob`，因为前者经过了更好的优化，可以加快训练
- 将其封装成一个便于调用的函数：


In [ ]:
def compute_example_loss(model, example, device):
    token_ids = example["token_ids"]
    prompt_len = example["prompt_len"]

    input_ids = torch.tensor(
        token_ids[:-1], dtype=torch.long, device=device
    ).unsqueeze(0)
    target_ids = torch.tensor(
        token_ids[1:], dtype=torch.long, device=device
    )

    logits = model(input_ids).squeeze(0)

    answer_start = max(prompt_len - 1, 0)
    answer_logits = logits[answer_start:]
    answer_targets = target_ids[answer_start:]

    loss = torch.nn.functional.cross_entropy(
        answer_logits, answer_targets
    )
    return loss

In [ ]:
with torch.no_grad():
    loss = compute_example_loss(
        model, train_examples[5730], device
    )

print(f"Loss: {loss:.2f}")

- 也可以并行计算多个样本，这称为批处理；不过为了降低资源需求，这里将其省略
- 附录 E 将讨论批处理


- 还可以定义一个包装函数，用于计算多个样本的损失：

In [ ]:
@torch.no_grad()
def evaluate_examples(model, examples, device):
    was_training = model.training
    model.eval()
    total_loss = 0.0
    num_examples = 0

    for example in examples:
        loss = compute_example_loss(model, example, device)
        total_loss += loss.item()
        num_examples += 1

    if was_training:
        model.train()

    return total_loss / num_examples

In [ ]:
train_loss = evaluate_examples(model, train_examples[:3], device)
print(f"Train loss (3 examples): {train_loss:.2f}")

In [ ]:
val_loss = evaluate_examples(model, val_examples[:3], device)
print(f"Validation loss (3 examples): {val_loss:.2f}")

- 训练期间将复用此函数跟踪进度；我们希望看到损失逐渐下降。

&nbsp;
## 8.7 实现蒸馏训练循环

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F14_raschka.webp" width=600>

- 训练循环与第 6 章几乎相同，区别在于现在会多次遍历训练集（多个训练轮次），并计算交叉熵损失而不是 GRPO 损失。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F15_raschka.webp" width=500>

In [ ]:
import time


def train_distillation(
    model,
    train_examples,
    val_examples,
    device,
    epochs=2,
    lr=5e-6,
    grad_clip_norm=None,
    seed=123,
    log_every=50,
    checkpoint_dir="checkpoints",
    csv_log_path=None,
):
    # 第 1 步：初始化优化器（模型已加载）
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()

    total_steps = epochs * len(train_examples)
    global_step = 0
    rng = random.Random(seed)

    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_distill_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    # 第 2 步：遍历训练轮次
    for epoch in range(1, epochs + 1):
        # 第 3 步：在训练轮次开始时打乱训练样本
        epoch_examples = list(train_examples)
        rng.shuffle(epoch_examples)

        # 第 4 步：遍历当前轮次中的训练样本
        for example in epoch_examples:
            global_step += 1

            # 第 5 步：重置损失梯度
            # （最佳实践是在每个步骤开始时执行此操作）
            optimizer.zero_grad()

            # 第 6 步：计算当前样本的交叉熵损失
            loss = compute_example_loss(model, example, device)

            # 第 7 步：反向传播梯度
            loss.backward()

            # 可选：裁剪过大的梯度以提高训练稳定性
            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), grad_clip_norm
                )

            # 第 8 步：更新模型权重
            optimizer.step()

            # 第 9 步：定期在验证集上评估当前模型
            if log_every and global_step % log_every == 0:
                val_loss = evaluate_examples(
                    model=model,
                    examples=val_examples,
                    device=device,
                )
                model.train()
                print(
                    f"[Epoch {epoch}/{epochs} "
                    f"Step {global_step}/{total_steps}] "
                    f"train_loss={loss.item():.4f} "
                    f"val_loss={val_loss:.4f}"
                )
                append_csv_metrics(
                    csv_log_path=csv_log_path,
                    epoch_idx=epoch,
                    total_steps=global_step,
                    train_loss=loss.item(),
                    val_loss=val_loss,
                )

        # 第 10 步：保存当前训练轮次的检查点
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=global_step,
            suffix=f"epoch{epoch}",
        )
        print(f"Saved checkpoint to {ckpt_path}")
    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-distill-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    epoch_idx,
    total_steps,
    train_loss,
    val_loss,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "epoch,total_steps,train_loss,val_loss\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{epoch_idx},{total_steps},{train_loss:.6f},"
            f"{val_loss:.6f}\n"
        )

- 序列长度设为 2048 时，本次训练约需 15 GB 内存
- 要降低资源需求，可以过滤掉较长序列；例如在之前的 `filtered_examples = filter_examples_by_max_len(examples, max_len=2048)` 步骤中，将 `max_len` 从 `2048` 改为 `1024` 或 `512`


In [ ]:
torch.manual_seed(0)

train_distillation(
    model,
    train_examples=train_examples[:10],
    val_examples=val_examples[:10],
    device=device,
    epochs=2,
    lr=5e-6,
    grad_clip_norm=1.0,  # 与第 6 章相同
    seed=123,
    log_every=5,
    csv_log_path="train_distill_metrics.csv"
)

- 上面的运行仅用于简短演示
- 只在 10 个样本（`train_examples[:10]`）上训练，并只在 10 个验证样本（`val_examples`）上评估
- 真正训练时应改为 `train_examples` 和 `val_examples`


&nbsp;
## 8.8 评估蒸馏后的模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F16_raschka.webp" width=500>

- 无需在此 notebook 中运行蒸馏训练，可以下载以下便捷脚本。

In [ ]:
# from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch08/04_train_with_distillation/distill.py"
)

```bash
uv run distill.py \
--data_path deepseek-r1-math-train.json \
--validation_size 25 \
--epochs 3 \
--lr 1e-5 \
--max_seq_len 2048 \
--use_think_tokens \
--grad_clip 1.0
```

- 使用上述设置时，训练在 DGX Spark 上约需 3 小时 5 分钟，并占用 15.02 GB GPU 内存
- 如果不想自行运行，下面将下载生成的指标文件进行分析


In [ ]:
download_from_github(
    "ch08/03_logs/deepseek-r1-2048_distill_metrics.csv"
)

In [ ]:
import csv
import matplotlib.pyplot as plt

def plot_distill_metrics(csv_path="train_distill_metrics.csv"):
    total_steps, train_losses, val_losses, epoch_bounds = [], [], [], {}

    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            step = int(row["total_steps"])
            epoch = int(row["epoch"])
            total_steps.append(step)
            train_losses.append(float(row["train_loss"]))
            val_losses.append(float(row["val_loss"]))
            epoch_bounds.setdefault(epoch, [step, step])[1] = step  # 每个训练轮次的第一步/最后一步

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(total_steps, train_losses, label="train_loss", alpha=0.3)
    ax.plot(total_steps, val_losses, label="val_loss")
    ax.set_xlabel("Total Steps")
    ax.set_ylabel("Loss")
    ax.legend()

    # 训练轮次坐标轴
    epoch_axis = ax.secondary_xaxis("bottom")
    epoch_axis.spines["bottom"].set_position(("outward", 45))
    epochs = sorted(epoch_bounds)
    epoch_axis.set_xticks(
        [(epoch_bounds[epoch][0] + epoch_bounds[epoch][1]) / 2 
          for epoch in epochs]
    )
    epoch_axis.set_xticklabels(epochs)
    epoch_axis.set_xlabel("Epoch")

    plt.tight_layout()
    plt.show()

plot_distill_metrics("deepseek-r1-2048_distill_metrics.csv")

- 训练损失基于单个样本，因此噪声较大
- 也可以像验证损失一样在多个样本（例如 25 个）上进行评估，但这会增加训练时间
- 我们主要关心验证损失，并希望看到它下降
- 验证损失先急剧下降，随后不再取得显著进展
- 可以提高学习率以更激进地训练，不过总体来看，验证曲线表现良好


- 与第 7 章类似，可以使用第 3 章的评估工具评估生成的模型检查点。

In [ ]:
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)

```bash
uv run evaluate_math500.py \
--dataset_size 500 \
--which_model reasoning \
--max_new_tokens 4096 \
--checkpoint_path run_11/checkpoints/distill/qwen3-0.6B-distill-step05746-epoch1.pth
```

|      | 教师数据                         | 轮次 | MATH-500 准确率 | 最终验证损失 |
| ---- | ------------------------------------ | ----- | ------------ | -------------- |
| 1    | 基础模型（第 3 章）                 | -     | 15.2%        | -              |
| 2    | 推理模型（第 3 章）                 | -     | 48.2%        | -              |
| 3    | DeepSeek R1 蒸馏数据                | 1     | 30.6%        | 0.5436         |
| 4    | DeepSeek R1 蒸馏数据                | 2     | 32.4%        | 0.5349         |
| 5    | DeepSeek R1 蒸馏数据                | 3     | 33.6%        | 0.5343         |
| 6    | Qwen3 235B A22B 蒸馏数据            | 1     | 45.0%        | 0.4043         |
| 7    | Qwen3 235B A22B 蒸馏数据            | 2     | 43.8%        | 0.3963         |
| 8    | Qwen3 235B A22B 蒸馏数据            | 3     | 44.2%        | 0.3948         |


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F18_raschka.webp" width=500>

&nbsp;
## 8.9 推理模型的未来方向

- 本节没有代码
- 更多参考资料另见附录 A


&nbsp;
## 8.10 结语

- 本节没有代码。

- 希望你喜欢本书！
- 正文到这里就结束了，但后面还有一些可能有用的“附录”章节（涵盖 LLM 架构细节、以批处理模式运行代码来提高吞吐量、不同评估方法的讨论等）
- 此外，还可以在[附加材料](https://github.com/rasbt/reasoning-from-scratch?tab=readme-ov-file#bonus-material)部分找到更多内容


- 我经常在博客 https://magazine.sebastianraschka.com 上撰写 AI 和 LLM 相关文章，希望这些内容对你有所帮助。

&nbsp;
## 总结

- 本节没有代码。